# 05 · 超参数搜索

用 `tuning/` 模块做网格 / 随机搜索。**目标函数由你注入**，
所以不需要 GPU 也能跑通整个流程（这里用解析式代理指标演示）。

## 1. 定义搜索空间

连续参数建议用**对数区间**（`loguniform`）：学习率跨 1e-5 ~ 1e-3 两个数量级，
均匀采样会把大量预算浪费在无效的大学习率上。整数维度同理用 `int_log`。

In [ ]:
import sys

sys.path.insert(0, "..")

from llm_finetuning.tuning import SearchSpace

space = SearchSpace.from_dict(
    {
        "learning_rate": ("loguniform", 1e-5, 1e-3),
        "lora.r": ("int_log", 4, 64),
        "batch_size": ("categorical", [4, 8, 16]),
        "warmup_ratio": ("uniform", 0.0, 0.2),
    }
)
print("维度:", space.names())
print("网格组合数:", space.grid_size())
print(space.describe())

## 2. 目标函数

真实场景里换成"训练 N 步 → 在验证集上取指标"。

铁律：搜索目标必须是**验证集指标**。用训练集指标选超参，等价于直接选出
过拟合最严重的那一组。

In [ ]:
import math


def objective(params):
    """代理指标：存在最优学习率与最优秩，越小越好。"""
    lr = params["learning_rate"]
    r = params["lora.r"]
    bs = params["batch_size"]
    warmup = params["warmup_ratio"]

    loss = (math.log10(lr) - math.log10(2e-4)) ** 2   # 最优 lr ≈ 2e-4
    loss += 0.5 * (math.log2(r) - math.log2(16)) ** 2  # 最优 r ≈ 16
    loss += 0.02 * abs(bs - 8)                          # 偏好 batch=8
    loss += 0.3 * warmup                                # 轻微偏好小 warmup
    return loss


print("lr=2e-4, r=16 时:", round(objective({"learning_rate": 2e-4, "lora.r": 16,
                                            "batch_size": 8, "warmup_ratio": 0.0}), 4))
print("lr=1e-3, r=64 时:", round(objective({"learning_rate": 1e-3, "lora.r": 64,
                                            "batch_size": 16, "warmup_ratio": 0.2}), 4))

In [ ]:
from llm_finetuning.tuning import HyperparameterSearcher

random_result = HyperparameterSearcher(
    space,
    objective,
    direction="minimize",
    max_trials=20,
    seed=42,
).search(strategy="random", n_trials=20)

print(f"试验次数: {len(random_result.trials)}")
print(f"最优得分: {random_result.best_score:.4f}")
print(f"最优参数: {random_result.best_params}")

## 3. 网格 vs 随机

同等预算下**随机搜索通常优于网格搜索**：网格会把后面的维度锁死在前几个取值上，
而随机搜索让每个维度都被充分探索。

> 项目里的网格搜索在 `max_trials` 截断时采用**均匀抽样**而非取前缀，
> 就是为了避免这个偏差 —— 早先版本取前缀时，同样预算下最优值 2.69，
> 改成均匀抽样后为 0.40。组合数超过 20000 时会自动退化为不重复随机采样，
> 避免笛卡尔积爆炸。

In [ ]:
grid_result = HyperparameterSearcher(
    space, objective, direction="minimize", max_trials=20
).search(strategy="grid")

print(f"网格最优: {grid_result.best_score:.4f}")
print(f"随机最优: {random_result.best_score:.4f}")
print("\nTop 3:")
for trial in random_result.top_k(3):
    print(f"  {trial.score:.4f}  {trial.params}")

## 4. 结果落盘与回写配置

`apply_params` 支持点号路径（`lora.r` 会写入 `config.lora_config.r`），
返回成功写入的参数名列表 —— 用它确认没有参数被静默丢弃。

In [ ]:
from llm_finetuning import Config, FinetuningMethod
from llm_finetuning.tuning import apply_params

config = Config(
    model_name_or_path="Qwen/Qwen2.5-0.5B-Instruct",
    finetuning_method=FinetuningMethod.LORA,
)
applied = apply_params(config, random_result.best_params)

print("已写入:", applied)
print("learning_rate      =", config.training.learning_rate)
print("lora_config.r      =", config.lora_config.r)

random_result.save_csv("./outputs/search_trials.csv")
print("\n试验记录已写入 ./outputs/search_trials.csv")